<a href="https://colab.research.google.com/github/alaycheme25/chemeng277_batteryproject/blob/main/machine_learning_3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pymatgen paretoset periodictable

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.decomposition import PCA
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import paretoset
import periodictable
from pymatgen.core import Structure
import ast

In [ ]:
pdata = pd.read_excel("MATSCI_176_Project_Data.xlsx")
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,structure,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,"{'@module': 'pymatgen.core.structure', '@class...",0.333634,0,417.933696,4.179337
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.744135,0.7074,1920.448911,19.204489
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.034716,0,735.871648,7.358716
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.330999,0,576.149771,5.761498
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,"{'@module': 'pymatgen.core.structure', '@class...",-0.415156,0,82.574983,0.825750


In [ ]:
structures = pdata["structure"]
print(structures.shape)
structure_dict_test = ast.literal_eval(structures[1])
structure_test = Structure.from_dict(structure_dict_test)
print(structure_test)
print(structure_test.lattice.abc)
print(structure_test.volume)

(5786,)
Full Formula (Li3 Sb1)
Reduced Formula: Li3Sb
abc   :   4.627959   4.627959   4.627959
angles:  60.000007  60.000010  60.000007
pbc   :       True       True       True
Sites (4)
  #  SP       a      b     c    magmom
---  ----  ----  -----  ----  --------
  0  Li    0.5    0.5   0.5          0
  1  Li    0.25   0.25  0.25        -0
  2  Li    0.75   0.75  0.75        -0
  3  Sb    0     -0     0            0
(4.627959209506349, 4.627958733729667, 4.62795882)
70.08959778250679


In [ ]:
def get_volume(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.volume
    except:
        return np.nan
structure_volums = pdata["structure"].apply(get_volume) # Volumes in Angstrom^3
def get_abc(structure_str):
    try:
        structure_dict = ast.literal_eval(structure_str)
        structure = Structure.from_dict(structure_dict)
        return structure.lattice.abc
    except:
        return np.nan
structure_abc = pdata["structure"].apply(get_abc) # abc values in Angstrom
structure_abc_df = pd.DataFrame(structure_abc.tolist(), columns=['a', 'b', 'c'])
pdata = pdata.drop(columns=["structure"], axis=1)

In [ ]:
structure_volums = structure_volums.to_frame()
structure_volums = structure_volums.fillna(0)
structure_volums = structure_volums["structure"]
print(structure_volums.shape)
print(structure_abc_df.shape)
pdata["structure_volume"] = structure_volums.values

(5786,)
(5786, 3)


In [ ]:
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L,structure_volume
0,Li0-3Ag,Li,['Ag'],1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,0.333634,0,417.933696,4.179337,103.086779
1,Li0-3Sb,Li,['Sb'],1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,-0.744135,0.7074,1920.448911,19.204489,70.089598
2,Li0-1Bi,Li,['Bi'],1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,-0.034716,0,735.871648,7.358716,58.627385
3,Li0-3Ce,Li,['Ce'],1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,-0.330999,0,576.149771,5.761498,1918.735425
4,Li0-3Ca,Li,['Ca'],1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,-0.415156,0,82.574983,0.825750,47.631359


In [ ]:
pdata = pdata.dropna()
pdata["elements"] = pdata["elements"].apply(lambda x: x.replace("[", "").replace("]", "").replace(" ", "").replace("'", ""))
pdata.head()

,battery_formula,working_ion,elements,nelements,formula_charge,formula_discharge,max_delta_volume,average_voltage,capacity_grav,capacity_vol,...,stability_charge,stability_discharge,electrode_density,material_density,working_ion_cost_per_kWh,formation_energy_per_atom,band_gap,energy_density,battery_cost_per_L,structure_volume
0,Li0-3Ag,Li,Ag,1.0,Ag,Li3Ag,2.736089,0.210565,624.785871,1984.816229,...,0.003609,0.000000,3.176794,2.592432,10.0,0.333634,0,417.933696,4.179337,103.086779
1,Li0-3Sb,Li,Sb,1.0,Sb,Li3Sb,1.569237,1.015953,563.913254,1890.293779,...,0.328351,0.000000,3.352100,3.378028,10.0,-0.744135,0.7074,1920.448911,19.204489,70.089598
2,Li0-1Bi,Li,Bi,1.0,Bi,LiBi,0.368255,0.796796,124.126099,923.538664,...,0.049545,0.000000,7.440326,2.237712,10.0,-0.034716,0,735.871648,7.358716,58.627385
3,Li0-3Ce,Li,Ce,1.0,Ce,Li3Ce,2.951183,-0.444846,499.595769,1295.168213,...,0.000000,0.333634,2.592432,2.583328,10.0,-0.330999,0,576.149771,5.761498,1918.735425
4,Li0-3Ca,Li,Ca,1.0,Ca,Li3Ca,1.513956,-0.064931,1320.248328,1271.741142,...,0.000000,0.057445,0.963259,7.527518,10.0,-0.415156,0,82.574983,0.825750,47.631359


In [ ]:
pdata_norm = pdata.iloc[:, 6:]
print(pdata_norm.columns)

Index(['max_delta_volume', 'average_voltage', 'capacity_grav', 'capacity_vol',
       'energy_grav', 'energy_vol', 'stability_charge', 'stability_discharge',
       'electrode_density', 'material_density', 'working_ion_cost_per_kWh',
       'formation_energy_per_atom', 'band_gap', 'energy_density',
       'battery_cost_per_L', 'structure_volume'],
      dtype='object')


In [ ]:
numeric_cols = [
    "material_density",
    "formation_energy_per_atom",
    "band_gap"
]

for col in numeric_cols:
    pdata_norm[col] = pd.to_numeric(pdata_norm[col], errors="coerce")

In [ ]:
pdata_norm = pdata_norm.dropna()

In [ ]:
# Get the energy grav, structure volume, battery cost, and bandgap energy from pdata_front and combine into a new dataframe
energy_grav = pdata_norm["energy_grav"]
cap_grav = pdata_norm["capacity_grav"]
battery_cost = pdata_norm["battery_cost_per_L"]
energy_density = pdata_norm["energy_density"]
voltage = pdata_norm['average_voltage']
training_data = pdata_norm.drop(columns=["energy_grav", "capacity_grav", "battery_cost_per_L", "energy_density", "average_voltage", "energy_vol", "capacity_vol", "electrode_density"])

In [ ]:
def scoring(e_g,c_g,b_c,e_d,v):
    e_g = e_g.to_numpy().astype(float)
    c_g = c_g.to_numpy().astype(float)
    b_c = b_c.to_numpy().astype(float)
    e_d = e_d.to_numpy().astype(float)
    v = v.to_numpy().astype(float)
    # w_i = w_i.to_numpy().astype(float)
    # print(e_g.shape, s_v.shape, b_c.shape, b_e.shape, w_i.shape)
    # print(type(e_g), type(s_v), type(b_c), type(b_e), type(w_i))
    score = abs(e_g) + abs(c_g) - abs(b_c) + abs(e_d) + abs(v)
    # score = score/np.max(score)
    # score = np.abs(e_d)/np.max(e_d) + np.abs(c_g*e_g/b_c)/(np.max(c_g)*np.max(e_g)/np.max(b_c)) + np.abs(v)/np.max(v)
    # score = score/np.max(score)
    return pd.Series(score)

In [ ]:
battery_score = scoring(energy_grav, cap_grav, battery_cost, energy_density, voltage)

fit_train, fit_test, score_train, score_test = train_test_split(
    training_data,
    battery_score,
    test_size=0.1,
    random_state=42
)

In [ ]:
norm_fit_train = pd.DataFrame(
    StandardScaler().fit_transform(fit_train),
    columns=fit_train.columns
)


In [ ]:
ridge = linear_model.Ridge(alpha=1.0).fit(norm_fit_train, score_train, sample_weight=None)
# norm_fit_test = pd.DataFrame(
#     StandardScaler().fit_transform(fit_test),
#     columns=fit_test.columns
# )

score_pred = ridge.score(norm_fit_test, score_test)
print(score_pred)

0.27559803187928533


In [ ]:
best = np.where(battery_score == np.max(battery_score))[0][0]  #np.min(battery_score) was changed to max(battery_score) to obtain best capacity
print(pdata.iloc[best])
print("Battery Score:", np.max(battery_score))

battery_formula              Li1-3V2CrO6
working_ion                           Li
elements                          V,Cr,O
nelements                            3.0
formula_charge                  LiV2CrO6
formula_discharge              Li3V2CrO6
max_delta_volume                0.014929
average_voltage                 2.985936
capacity_grav                 198.017212
capacity_vol                  790.405191
energy_grav                   591.266759
energy_vol                   2360.099459
stability_charge                0.106473
stability_discharge              0.00128
electrode_density               3.991598
material_density                 5.70676
working_ion_cost_per_kWh            10.0
formation_energy_per_atom      -1.853804
band_gap                          0.3149
energy_density               2360.099459
battery_cost_per_L             23.600995
structure_volume              242.720605
Name: 3207, dtype: object
Battery Score: 18374.29795743876


In [ ]:
X = training_data
y = battery_score
y = y.fillna(0)
y = y.astype(float)

In [ ]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", linear_model.Ridge(alpha=3.0))
])

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
MAE = []
MSE = []
bestmaterial = []

for train_idx, test_idx in kf.split(X):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    #Find best material in predicted set
    bestmaterial.append(X_test.iloc[np.where(y_pred == np.max(y_pred))[0][0]])
    print(np.max(y_pred))

    r2_scores.append(r2_score(y_test, y_pred))
    MSE.append(mean_squared_error(y_test, y_pred))
    MAE.append(mean_absolute_error(y_test, y_pred))

print("Mean R2:", np.mean(r2_scores))
print("Mean MSE:", np.mean(MSE))
print("Mean MAE:", np.mean(MAE))
print("Best materials in each fold:", type(bestmaterial[0]))
bestmaterial_df = pd.DataFrame(bestmaterial)

17320.94508911402
13310.348984644133
8739.909619875943
20473.09313843158
7199.759284578825
Mean R2: 0.2630076484355701
Mean MSE: 1601805.0469765267
Mean MAE: 945.5525349497486
Best materials in each fold: <class 'pandas.core.series.Series'>


In [ ]:
for i in range(len(bestmaterial)):
    best_material = bestmaterial_df.iloc[i]
    min_diff = float('inf')
    best_match_index = -1
    for j in range(len(pdata)):
        diff = np.sum(np.abs(best_material - pdata.iloc[j, 6:6+len(best_material)]))
        if diff < min_diff:
            min_diff = diff
            best_match_index = j
    print(f"Best material in fold {i+1}:")
    print(pdata.iloc[best_match_index, 0])

Best material in fold 1:
Ca0.25-0.5C
Best material in fold 2:
Li1-3Ti2(PO4)3
Best material in fold 3:
Mg0-3C
Best material in fold 4:
Ca0-3N2
Best material in fold 5:
Li0-3ClO


In [ ]:

from sklearn.linear_model import ElasticNet

# X = training_data (your features)
# y = battery_score (your target)

#replace ridge with elasticnet
# Elastic Net model (pick starting values; tune later)
alpha = 1.0
l1_ratio = 0.5  # 0=ridge-like, 1=lasso-like

enet_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000, random_state=42))
])

enet_pipeline.fit(X_train, y_train)

# R^2 score on test set (same as your ridge.score())
score_pred = enet_pipeline.score(X_test, y_test)
print("ElasticNet test R^2:", score_pred)

In [ ]:
#replace Ridge KFold with Elastic Net KFold

X = training_data
y = battery_score.fillna(0).astype(float)

# Elastic Net hyperparameters
alpha = 1.0
l1_ratio = 0.5

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000, random_state=42))
])

kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
MAE = []
MSE = []
bestmaterial = []

for train_idx, test_idx in kf.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # store best predicted sample in this fold (same idea as your ridge code)
    bestmaterial.append(X_test.iloc[np.argmax(y_pred)])

    r2_scores.append(r2_score(y_test, y_pred))
    MSE.append(mean_squared_error(y_test, y_pred))
    MAE.append(mean_absolute_error(y_test, y_pred))

print("Mean R^2:", np.mean(r2_scores))
print("Mean MSE:", np.mean(MSE))
print("Mean MAE:", np.mean(MAE))

bestmaterial_df = pd.DataFrame(bestmaterial)
bestmaterial_df